In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from google import genai

In [ ]:
#Load API key
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", api_key is not None)

client = genai.Client(api_key=api_key)

API key loaded: True


In [ ]:
#Load the csv file
rules_df = pd.read_csv("Association_rules.csv")
rules_df.columns = [c.strip() for c in rules_df.columns]
print(rules_df.shape)
rules_df.head()

(5278, 10)


,No.,Premises,Conclusion,Support,Confidence,Laplace,Gain,p-s,Lift,Conviction
0,986,"Java, JavaScript","AWS, TypeScript",0.017907,0.370100,0.970930,-0.078860,0.016699,14.825589,1.547923
1,987,TypeScript,"Python, Docker",0.014202,0.370115,0.976723,-0.062541,0.011704,5.685377,1.484240
2,988,TypeScript,"Python, Java, AWS",0.014202,0.370115,0.976723,-0.062541,0.011919,6.220620,1.493133
3,989,"Python, Hadoop","Scala, Kafka",0.010056,0.370130,0.983340,-0.044282,0.009680,26.725970,1.565642
4,990,"Kubernetes, Go","AWS, Agile",0.012835,0.370229,0.978899,-0.056499,0.011193,7.815831,1.512662


In [ ]:
#Filter rules by skill
def split_skills(cell):
    if pd.isna(cell):
        return []
    return [s.strip().lower() for s in str(cell).split(",")]


def get_relevant_rules(current_skills, rules_df, top_n=8):
    current_skills_lower = set(s.strip().lower() for s in current_skills)

    def overlaps(premises_cell):
        premise_skills = set(split_skills(premises_cell))
        return len(premise_skills & current_skills_lower) > 0

    matched = rules_df[rules_df["Premises"].apply(overlaps)].copy()
    matched = matched.sort_values(by=["Confidence", "Lift"], ascending=False)
    return matched.head(top_n)


test_skills = ["SQL", "Excel"]
get_relevant_rules(test_skills, rules_df)

,No.,Premises,Conclusion,Support,Confidence,Laplace,Gain,p-s,Lift,Conviction
5268,6254,"SQL, Kubernetes, Open Source RDBMS",Docker,0.012967,1.000000,1.000000,-0.012967,0.011678,10.063471,inf
5269,6255,"SQL, Docker, HTML/CSS",Kubernetes,0.011026,1.000000,1.000000,-0.011026,0.009847,9.349691,inf
5266,6252,"SQL, Go, Scala",Java,0.013320,1.000000,1.000000,-0.013320,0.011017,5.785404,inf
5259,6245,"SQL, Go, TypeScript",Python,0.014908,1.000000,1.000000,-0.014908,0.010442,3.338684,inf
5247,6233,"SQL, Kubernetes, HTML/CSS",Docker,0.011026,0.996016,0.999956,-0.011115,0.009926,10.023377,226.058307
5245,6231,"SQL, Machine Learning, R",Python,0.010188,0.995690,0.999956,-0.010277,0.007124,3.324293,162.511534
5238,6224,"SQL, Go, Scala",Python,0.013232,0.993377,0.999913,-0.013408,0.009242,3.316573,105.772593
5219,6205,"SQL, Docker, RDBMS",Java,0.010277,0.987288,0.999869,-0.010541,0.008477,5.711861,65.069231


In [5]:
#Build the prompt for the AI model
def build_prompt(current_skills, target_role, relevant_rules):
    rules_text = "\n".join(
        f"- If someone has [{row['Premises']}], job postings often also require [{row['Conclusion']}] "
        f"(confidence={row['Confidence']:.2f}, lift={row['Lift']:.2f})"
        for _, row in relevant_rules.iterrows()
    )

    prompt = f"""You are a career advisor for IT students.

A student currently has these skills: {', '.join(current_skills)}.
Their target job role is: {target_role}.

Here are real skill co-occurrence patterns found in IT job postings, from association rule mining:
{rules_text}

Based on this evidence, recommend the 3 to 5 most important skills the student should learn next.
For each skill, give a one-sentence, plain-language reason tied to the patterns above.
Keep the whole answer short, friendly, and easy for a student to read — no jargon, no long paragraphs.
"""
    return prompt

In [12]:
#Call Gemini
def get_recommendation(current_skills, target_role, rules_df, top_n=8):
    relevant = get_relevant_rules(current_skills, rules_df, top_n=top_n)

    if relevant.empty:
        return "No matching skill patterns were found for these inputs. Try different or more general current skills."

    prompt = build_prompt(current_skills, target_role, relevant)

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
    )
    return response.text

In [13]:
result_1 = get_recommendation(
    current_skills=["SQL", "Excel"],
    target_role="Data Analyst",
    rules_df=rules_df,
)
print(result_1)

Hi there! Since you already have a great foundation in SQL and Excel, you are in a fantastic position to build up your Data Analyst toolkit. 

Based on real job posting data, here are the top **4 skills** you should learn next:

* **Python:** Job patterns show that when you combine SQL with statistical tools, employers almost always expect you to know Python.
* **R:** Our data shows that pairing R alongside SQL creates a powerful combination that frequently leads to top data analysis job requirements.
* **Machine Learning:** Job listings reveal that adding Machine Learning to your SQL background strongly unlocks advanced, high-demand data roles.
* **RDBMS (Relational Database Management Systems):** Deepening your SQL knowledge to understand general database systems is a key stepping stone that consistently pairs with modern data infrastructure jobs.

Focusing on these will turn your current foundation into a complete, job-ready data analysis skill set. You've got this!


In [14]:
result_2 = get_recommendation(
    current_skills=["Html"],
    target_role="Programmer",
    rules_df=rules_df,
)
print(result_2)

Hello! It's great that you already have HTML under your belt. Based on real hiring data from tech job postings, here are the top 4 skills you should learn next to become a well-rounded programmer:

*   **CSS:** Since 82% of job postings that require HTML also look for CSS, learning this skill is the most natural next step to help you style and design web pages.
*   **JavaScript:** Once you combine HTML and CSS, nearly 80% of employers will expect you to know JavaScript to make your websites interactive and dynamic.
*   **Java:** Adding Java to your web skills opens up full-stack programming roles, as over 85% of job posts mentioning Java and web tools require the full combination.
*   **SQL:** Job postings that pair HTML with backend database skills like SQL consistently require web knowledge, making SQL a fantastic skill for working with data behind the scenes.

Focusing on **CSS** and **JavaScript** first will give you the strongest immediate boost! Good luck on your learning journey